# Students Streaming Test
Run each cell once, in order, in Synapse PySpark with Delta support and writable default storage. Start with no tables named `students_source` or `students_target` in the current database; CREATE intentionally fails if they already exist.

The first stream copies 3 students. After inserting 2 more, the identical stream reuses the same checkpoint and copies only those 2. `availableNow=True` stops each stream when available data is processed. Do not rerun setup or change the checkpoint between passes.

In [ ]:
from uuid import uuid4

spark.sql('CREATE TABLE students_source (id INT, studentname STRING) USING DELTA')
spark.sql('CREATE TABLE students_target (id INT, studentname STRING) USING DELTA')
checkpoint_path = f"{spark.conf.get('spark.sql.warehouse.dir').rstrip('/')}/students_checkpoint_{uuid4().hex}"

In [ ]:
spark.sql("INSERT INTO students_source VALUES (1, 'Alice'), (2, 'Bob'), (3, 'Carol')")

In [ ]:
query = (
    spark.readStream.table('students_source')
    .writeStream.format('delta').outputMode('append')
    .option('checkpointLocation', checkpoint_path)
    .trigger(availableNow=True)
    .toTable('students_target')
)
query.awaitTermination()

In [ ]:
display(spark.sql('SELECT * FROM students_target ORDER BY id'))

In [ ]:
spark.sql("INSERT INTO students_source VALUES (4, 'David'), (5, 'Emma')")

In [ ]:
query = (
    spark.readStream.table('students_source')
    .writeStream.format('delta').outputMode('append')
    .option('checkpointLocation', checkpoint_path)
    .trigger(availableNow=True)
    .toTable('students_target')
)
query.awaitTermination()

In [ ]:
display(spark.sql('SELECT * FROM students_source ORDER BY id'))
display(spark.sql('SELECT * FROM students_target ORDER BY id'))
display(spark.sql("""
    SELECT 'source' AS table_name, COUNT(*) AS row_count FROM students_source
    UNION ALL
    SELECT 'target' AS table_name, COUNT(*) AS row_count FROM students_target
"""))